# 🔬 AI Detector — Fase 4: Evolução com Dados Massivos e GPU

**Objetivo:** Elevar a acurácia de **93,4% → 96%+** usando:
- 🗄️ Dados reais de benchmarks acadêmicos (RAID, MAGE, HC3, AI Detection Pile)
- ⚙️ 8 novas features estatísticas/estilométricas (12 → 20 features)
- ⚡ XGBoost + LightGBM com suporte a GPU (CUDA)
- 🔁 Otimização automática de hiperparâmetros com Optuna
- 🧩 Ensemble calibrado (voting + stacking)

---

## Roadmap do Notebook

| Seção | Conteúdo |
|---|---|
| 0 | Setup, GPU check, imports |
| 1 | Carregamento dos datasets do HuggingFace |
| 2 | EDA — Análise exploratória dos dados |
| 3 | Baseline — Re-avaliação do modelo v3 atual |
| 4 | Feature Engineering Phase 4 (8 novas features) |
| 5 | Extração paralela com CPU multi-core |
| 6 | Seleção de features (importância + correlação) |
| 7 | Treinamento XGBoost + LightGBM (GPU) |
| 8 | Optuna: busca de hiperparâmetros |
| 9 | Ensemble calibrado |
| 10 | Avaliação completa (accuracy, F1, AUC, calibração) |
| 11 | Análise de erros |
| 12 | Salvar modelo v4 e atualizar o serviço |

---

**Datasets utilizados:**

| Dataset | HuggingFace ID | Tamanho | Licença |
|---|---|---|---|
| RAID (ACL 2024) | `liamdugan/raid` | 7.4M+ | MIT |
| AI Detection Pile | `artem9k/ai-text-detection-pile` | 1.39M | MIT |
| HC3 | `Hello-SimpleAI/HC3` | 48k | CC-BY-SA |
| MAGE (ACL 2024) | `yaful/MAGE` | 437k | Apache 2.0 |

---
## Seção 0 — Setup, GPU Check e Imports

Instalamos as dependências necessárias e verificamos se há GPU disponível.
O XGBoost 3.x usa `device='cuda'` para GPU. O LightGBM usa `device='gpu'`.
Se não houver GPU, todo o código roda normalmente na CPU (apenas mais lento).

In [ ]:
# ── Instalação de dependências ───────────────────────────────────────────────
import subprocess, sys

packages = [
    "datasets>=2.14",
    "xgboost>=2.0",
    "lightgbm>=4.0",
    "optuna>=3.0",
    "nltk>=3.8",
    "imbalanced-learn>=0.11",
    "shap>=0.44",
    "scikit-learn>=1.4",
    "joblib>=1.3",
    "matplotlib>=3.7",
    "seaborn>=0.13",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("✓ Dependências instaladas")

In [ ]:
# ── Imports principais ───────────────────────────────────────────────────────
import os, sys, json, re, time, warnings
from pathlib import Path
from collections import Counter
from math import log2, sqrt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, brier_score_loss
)
from sklearn.pipeline import Pipeline

import xgboost as xgb
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
print("✓ Imports concluídos")

# ── Adiciona raiz do projeto ao path para importar funções existentes ────────
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
MODELS_DIR  = ROOT / "app" / "models" / "ml"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Raiz do projeto: {ROOT}")

In [ ]:
# ── Detecção de GPU ──────────────────────────────────────────────────────────
#
# XGBoost 3.x: usa device='cuda' ou device='cpu'
# LightGBM 4.x: usa device='gpu' ou device='cpu'
# Se a GPU não estiver disponível, o código continua na CPU sem erros.

def _check_xgb_gpu() -> bool:
    """Testa se o XGBoost consegue usar CUDA."""
    try:
        m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2, verbosity=0)
        m.fit(np.random.rand(50, 5), np.random.randint(0, 2, 50))
        return True
    except Exception:
        return False

def _check_lgb_gpu() -> bool:
    """Testa se o LightGBM consegue usar GPU."""
    try:
        m = lgb.LGBMClassifier(device='gpu', n_estimators=2, verbose=-1)
        m.fit(np.random.rand(50, 5), np.random.randint(0, 2, 50))
        return True
    except Exception:
        return False

USE_XGB_GPU = _check_xgb_gpu()
USE_LGB_GPU = _check_lgb_gpu()

XGB_DEVICE = 'cuda' if USE_XGB_GPU else 'cpu'
LGB_DEVICE = 'gpu'  if USE_LGB_GPU else 'cpu'

print(f"╔══════════════════════════════╗")
print(f"║  XGBoost  → device={XGB_DEVICE:4s}       ║")
print(f"║  LightGBM → device={LGB_DEVICE:4s}       ║")
print(f"╚══════════════════════════════╝")

---
## Seção 1 — Carregamento dos Datasets do HuggingFace

Utilizamos 4 datasets complementares:

1. **RAID** (`liamdugan/raid`) — Benchmark ACL 2024. Cobre 11 modelos (GPT-4, Llama-2, Mistral, Cohere, MPT), 11 gêneros textuais e 4 estratégias de decodificação. É o padrão ouro atual. Amostramos 50k por classe para manter o treino manejável.

2. **AI Detection Pile** (`artem9k/ai-text-detection-pile`) — 1.39M de amostras mistas. Boa cobertura de textos longos (Reddit, WebText). Amostramos 80k por classe.

3. **HC3** (`Hello-SimpleAI/HC3`) — Dataset Q&A pareado (pergunta humana + resposta ChatGPT). Alta qualidade, baixo ruído. Usamos completo (~48k).

4. **MAGE** (`yaful/MAGE`) — 27 LLMs diferentes, 10 domínios. Amostramos 40k por classe.

**Total alvo: ~300k amostras** após sampling estratificado e deduplicação.

In [ ]:
from datasets import load_dataset, Dataset

# ── Configurações de amostragem ──────────────────────────────────────────────
SEED           = 42
MAX_TEXT_LEN   = 4000   # caracteres — trunca textos muito longos
MIN_TEXT_LEN   = 50     # mínimo para análise de features

SAMPLE_RAID    = 50_000   # por classe (0=human, 1=ai)
SAMPLE_PILE    = 80_000   # por classe
SAMPLE_HC3     = None     # usa tudo
SAMPLE_MAGE    = 40_000   # por classe

rng = np.random.default_rng(SEED)


def _sample_df(df: pd.DataFrame, n: int | None, label_col='label') -> pd.DataFrame:
    """Amostragem estratificada por classe com seed fixo."""
    if n is None or len(df) <= n * 2:
        return df
    parts = []
    for lbl in df[label_col].unique():
        sub = df[df[label_col] == lbl]
        parts.append(sub.sample(min(n, len(sub)), random_state=SEED))
    return pd.concat(parts).reset_index(drop=True)


def _clean_text(t: str) -> str | None:
    """Remove espaços excessivos e trunca pelo limite máximo."""
    if not isinstance(t, str):
        return None
    t = re.sub(r'\s+', ' ', t).strip()
    if len(t) < MIN_TEXT_LEN:
        return None
    return t[:MAX_TEXT_LEN]


dfs = []  # acumula todos os DataFrames

print("Carregando datasets... (isso pode levar alguns minutos na primeira execução)")

In [ ]:
# ── 1. RAID ──────────────────────────────────────────────────────────────────
#
# Colunas relevantes: 'generation' (texto), 'label' (0=human, 1=ai),
# 'model' (qual LLM gerou), 'domain' (gênero textual)

print("[1/4] RAID...")
raid_raw = load_dataset("liamdugan/raid", split="train", trust_remote_code=True)
raid_df  = raid_raw.to_pandas()

# Normaliza colunas — RAID usa 'label' já como 0/1
raid_df = raid_df.rename(columns={'generation': 'text'})
raid_df['text']   = raid_df['text'].apply(_clean_text)
raid_df['source'] = 'raid'
raid_df = raid_df[['text', 'label', 'source']].dropna()
raid_df = _sample_df(raid_df, SAMPLE_RAID)

dfs.append(raid_df)
vc = raid_df['label'].value_counts()
print(f"   human={vc.get(0,0):,}  ai={vc.get(1,0):,}  total={len(raid_df):,}")

In [ ]:
# ── 2. AI Detection Pile ──────────────────────────────────────────────────────
#
# Colunas: 'text', 'label' (0=human, 1=ai)
# Fonte de humanos: Reddit, OpenAI WebText | IA: GPT-2, GPT-3, ChatGPT

print("[2/4] AI Detection Pile...")
pile_raw = load_dataset("artem9k/ai-text-detection-pile", split="train", trust_remote_code=True)
pile_df  = pile_raw.to_pandas()

pile_df['text']   = pile_df['text'].apply(_clean_text)
pile_df['label']  = pile_df['label'].astype(int)
pile_df['source'] = 'pile'
pile_df = pile_df[['text', 'label', 'source']].dropna()
pile_df = _sample_df(pile_df, SAMPLE_PILE)

dfs.append(pile_df)
vc = pile_df['label'].value_counts()
print(f"   human={vc.get(0,0):,}  ai={vc.get(1,0):,}  total={len(pile_df):,}")

In [ ]:
# ── 3. HC3 ────────────────────────────────────────────────────────────────────
#
# HC3 tem formato diferente: cada linha tem listas de respostas humanas e IA.
# Precisamos explodir essas listas em linhas individuais.

print("[3/4] HC3...")
hc3_raw = load_dataset("Hello-SimpleAI/HC3", "all", split="train", trust_remote_code=True)
hc3_df  = hc3_raw.to_pandas()

rows = []
for _, row in hc3_df.iterrows():
    for hr in (row.get('human_answers') or []):
        t = _clean_text(hr)
        if t: rows.append({'text': t, 'label': 0, 'source': 'hc3'})
    for ar in (row.get('chatgpt_answers') or []):
        t = _clean_text(ar)
        if t: rows.append({'text': t, 'label': 1, 'source': 'hc3'})

hc3_flat = pd.DataFrame(rows)
dfs.append(hc3_flat)
vc = hc3_flat['label'].value_counts()
print(f"   human={vc.get(0,0):,}  ai={vc.get(1,0):,}  total={len(hc3_flat):,}")

In [ ]:
# ── 4. MAGE ───────────────────────────────────────────────────────────────────
#
# MAGE: 27 LLMs, 10 domínios. Colunas: 'text', 'label' (0=human,1=ai)

print("[4/4] MAGE...")
mage_raw = load_dataset("yaful/MAGE", split="train", trust_remote_code=True)
mage_df  = mage_raw.to_pandas()

mage_df['text']   = mage_df['text'].apply(_clean_text)
mage_df['label']  = mage_df['label'].astype(int)
mage_df['source'] = 'mage'
mage_df = mage_df[['text', 'label', 'source']].dropna()
mage_df = _sample_df(mage_df, SAMPLE_MAGE)

dfs.append(mage_df)
vc = mage_df['label'].value_counts()
print(f"   human={vc.get(0,0):,}  ai={vc.get(1,0):,}  total={len(mage_df):,}")

In [ ]:
# ── Concatenar + Deduplicar + Embaralhar ─────────────────────────────────────
#
# Deduplicamos pelos primeiros 200 caracteres do texto para remover cópias
# entre datasets (alguns textos aparecem em mais de uma fonte).

df_all = pd.concat(dfs, ignore_index=True)
before = len(df_all)
df_all['text_key'] = df_all['text'].str[:200]
df_all = df_all.drop_duplicates(subset='text_key').drop(columns='text_key')
df_all = df_all.sample(frac=1, random_state=SEED).reset_index(drop=True)

after = len(df_all)
vc    = df_all['label'].value_counts()

print(f"\n{'='*50}")
print(f" Dataset consolidado")
print(f"{'='*50}")
print(f" Total bruto:    {before:>8,}")
print(f" Após dedup:     {after:>8,}  ({before-after:,} removidos)")
print(f" Human (0):      {vc.get(0,0):>8,}")
print(f" IA    (1):      {vc.get(1,0):>8,}")
print(f" Balanço:        {vc.get(1,0)/len(df_all)*100:.1f}% IA")
print(f"{'='*50}")

print("\nDistribuição por fonte:")
print(df_all.groupby(['source','label']).size().unstack(fill_value=0))

---
## Seção 2 — EDA: Análise Exploratória dos Dados

Antes de treinar, entendemos o que os dados nos dizem:
- Distribuição de tamanho de textos por classe
- Palavras mais frequentes (humano vs. IA)
- Como as features do modelo v3 se distribuem em cada classe

In [ ]:
# ── Distribuição de comprimento dos textos ───────────────────────────────────

df_all['text_len'] = df_all['text'].str.len()
df_all['word_count'] = df_all['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for lbl, name, color in [(0,'Humano','#22c55e'), (1,'IA','#ef4444')]:
    subset = df_all[df_all['label'] == lbl]
    axes[0].hist(subset['text_len'].clip(0, 4000), bins=60, alpha=0.6,
                 label=name, color=color, density=True)
    axes[1].hist(subset['word_count'].clip(0, 700), bins=60, alpha=0.6,
                 label=name, color=color, density=True)

axes[0].set_title('Distribuição: comprimento em caracteres', fontsize=12)
axes[0].set_xlabel('Caracteres'); axes[0].legend()
axes[1].set_title('Distribuição: contagem de palavras', fontsize=12)
axes[1].set_xlabel('Palavras'); axes[1].legend()
plt.tight_layout(); plt.show()

print("\nEstatísticas de comprimento:")
print(df_all.groupby('label')['word_count'].describe().round(1))

In [ ]:
# ── Top palavras exclusivas por classe ───────────────────────────────────────
#
# Calculamos quais palavras aparecem proporcionalmente muito mais em textos
# de IA do que em textos humanos e vice-versa. Isso é uma "análise de log-odds".

from collections import Counter
import string

STOP = {'the','a','an','is','in','of','to','and','or','it','that','this',
        'for','on','with','as','at','by','from','be','was','are','has','have',
        'not','but','can','do','we','he','she','they','you','i','my','his',
        'her','its','our','their','will','would','could','should','may','might'}

def word_freq(texts):
    c = Counter()
    for t in texts:
        words = re.findall(r'\b[a-z]{3,}\b', t.lower())
        c.update(w for w in words if w not in STOP)
    return c

human_freq = word_freq(df_all[df_all['label']==0]['text'].sample(20000, random_state=SEED))
ai_freq    = word_freq(df_all[df_all['label']==1]['text'].sample(20000, random_state=SEED))

total_h = sum(human_freq.values())
total_a = sum(ai_freq.values())

log_odds = {}
for w in set(human_freq) | set(ai_freq):
    ph = (human_freq.get(w, 0) + 1) / total_h
    pa = (ai_freq.get(w, 0) + 1) / total_a
    log_odds[w] = np.log(pa / ph)

sorted_lo = sorted(log_odds.items(), key=lambda x: x[1])
top_human = sorted_lo[:15][::-1]
top_ai    = sorted_lo[-15:]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
words_h, vals_h = zip(*top_human)
words_a, vals_a = zip(*top_ai)

axes[0].barh(words_h, [-v for v in vals_h], color='#22c55e', alpha=0.8)
axes[0].set_title('Palavras mais "humanas"', fontsize=12)
axes[0].set_xlabel('Log-odds favorecendo humano')

axes[1].barh(words_a, vals_a, color='#ef4444', alpha=0.8)
axes[1].set_title('Palavras mais usadas por IA', fontsize=12)
axes[1].set_xlabel('Log-odds favorecendo IA')

plt.tight_layout(); plt.show()

---
## Seção 3 — Baseline: Re-avaliação do Modelo v3

Antes de qualquer melhoria, medimos o desempenho atual do modelo v3 (93,4% em CV interno)
nos **novos dados externos** — isso revela se o modelo generaliza bem ou overfita.

Computamos as 12 features atuais em uma amostra de 10k exemplos do novo dataset.

In [ ]:
# ── Importa as funções de feature do serviço existente ───────────────────────

from app.services.detection_service import (
    extract_features, FEATURE_NAMES, N_FEATURES_V3,
    analyze_with_cascade
)

print(f"Features v3 ({N_FEATURES_V3} total):")
for i, name in enumerate(FEATURE_NAMES):
    print(f"  {i:2d}. {name}")

In [ ]:
# ── Extração de features para avaliação baseline ─────────────────────────────
#
# Amostramos 10k exemplos para a avaliação rápida do baseline.
# Usamos joblib.Parallel para processar em múltiplos cores da CPU.

from joblib import Parallel, delayed

BASELINE_SAMPLE = 10_000

df_baseline = df_all.sample(BASELINE_SAMPLE, random_state=SEED).reset_index(drop=True)

def _extract_safe(text: str) -> list[float]:
    """Extrai features com tratamento de exceções."""
    try:
        return extract_features(text, n_features=N_FEATURES_V3)
    except Exception:
        return [0.0] * N_FEATURES_V3

print(f"Extraindo {N_FEATURES_V3} features de {BASELINE_SAMPLE:,} textos...")
t0 = time.time()

feats_baseline = Parallel(n_jobs=-1, verbose=0)(
    delayed(_extract_safe)(t) for t in df_baseline['text']
)

X_baseline = np.array(feats_baseline, dtype=np.float32)
y_baseline = df_baseline['label'].values

print(f"✓ Concluído em {time.time()-t0:.1f}s  |  shape: {X_baseline.shape}")

In [ ]:
# ── Avalia modelo v3 nos novos dados ─────────────────────────────────────────

V3_MODEL_PATH = MODELS_DIR / 'detector_v3_rf.joblib'

if V3_MODEL_PATH.exists():
    rf_v3 = joblib.load(V3_MODEL_PATH)
    y_pred_v3 = rf_v3.predict(X_baseline)
    y_prob_v3 = rf_v3.predict_proba(X_baseline)[:, 1]

    acc_v3 = accuracy_score(y_baseline, y_pred_v3)
    f1_v3  = f1_score(y_baseline, y_pred_v3, average='weighted')
    auc_v3 = roc_auc_score(y_baseline, y_prob_v3)

    print(f"Modelo v3 nos dados externos:")
    print(f"  Accuracy : {acc_v3:.4f}  ({acc_v3*100:.2f}%)")
    print(f"  F1 (wtd) : {f1_v3:.4f}")
    print(f"  ROC-AUC  : {auc_v3:.4f}")
    print()
    print(classification_report(y_baseline, y_pred_v3, target_names=['Human','AI']))
else:
    print("⚠ Modelo v3 não encontrado. Pule para a Seção 7.")
    acc_v3 = 0.0

---
## Seção 4 — Feature Engineering: Fase 4 (8 Novas Features)

As novas features cobrem dimensões linguísticas que o modelo v3 não captura:

| # | Feature | Intuição |
|---|---|---|
| 12 | `readability_fog` | Índice Fog: % palavras longas (3+ sílabas). IA tende a usar vocabulário mais elaborado. |
| 13 | `stopword_ratio` | Proporção de stopwords. Humanos usam mais conectivos naturais. |
| 14 | `sentence_length_variance` | Variância (não CV) dos comprimentos de frase. Diferente do burstiness. |
| 15 | `comma_density` | Vírgulas por frase. IA cria listas com vírgulas muito regularmente. |
| 16 | `unique_trigrams_ratio` | Trigrams únicos / total. Mede diversidade de frases curtas. |
| 17 | `pos_noun_ratio` | Proporção de substantivos (POS tagging). Textos de IA têm mais substantivos. |
| 18 | `coherence_score` | Similaridade TF-IDF entre frases adjacentes. IA é mais coerente/monótona. |
| 19 | `exclamation_ratio` | Proporção de frases com exclamação. Humanos são mais expressivos. |

In [ ]:
# ── Feature 12: Readability Fog Index (proxy) ────────────────────────────────
#
# Gunning Fog = 0.4 * (palavras/frases + 100 * palavras_longas/palavras)
# Usamos apenas a componente de palavras longas (>= 7 chars como proxy de 3 sílabas)
# normalizada pelo total de palavras.

def compute_readability_fog(text: str) -> float:
    """
    Proporção de palavras 'complexas' (>= 7 caracteres) como proxy do Gunning Fog.
    IA tende a usar vocabulário mais elaborado → score mais alto.
    Retorna valor em [0.0, 1.0].
    """
    words = re.findall(r'\b[a-zA-Z]+\b', text)
    if not words:
        return 0.0
    long_words = sum(1 for w in words if len(w) >= 7)
    return long_words / len(words)


# ── Feature 13: Stopword Ratio ───────────────────────────────────────────────
#
# Stopwords são palavras gramaticais muito comuns ('the','is','of','a'...).
# Humanos tendem a ter stopword_ratio levemente maior pois escrevem de forma
# mais conversacional, usando mais 'and', 'so', 'but', 'I think'...

from nltk.corpus import stopwords as nltk_stopwords
try:
    _SW_EN = set(nltk_stopwords.words('english'))
except:
    nltk.download('stopwords', quiet=True)
    _SW_EN = set(nltk_stopwords.words('english'))

def compute_stopword_ratio(text: str) -> float:
    """
    Proporção de stopwords no texto (em relação ao total de palavras).
    Humanos: levemente mais alto (escrita mais informal/conversacional).
    Retorna valor em [0.0, 1.0].
    """
    words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    if not words:
        return 0.0
    sw_count = sum(1 for w in words if w in _SW_EN)
    return sw_count / len(words)


# ── Feature 14: Sentence Length Variance ────────────────────────────────────
#
# Diferente do Burstiness (que usa coeficiente de variação = std/mean),
# aqui usamos a variância normalizada pela média quadrática.
# Captura dispersão absoluta, sensível a outliers de frases muito longas/curtas.

def compute_sentence_length_variance(text: str) -> float:
    """
    Variância das comprimentos de frase normalizada pela média².
    Humanos tendem a misturar frases muito curtas e muito longas → variância alta.
    """
    from nltk.tokenize import sent_tokenize
    sentences = sent_tokenize(text)
    if len(sentences) < 2:
        return 0.0
    lengths = [len(s.split()) for s in sentences]
    mean_l  = np.mean(lengths)
    if mean_l < 1:
        return 0.0
    return float(np.var(lengths) / (mean_l ** 2))


print("✓ Features 12–14 definidas")

In [ ]:
# ── Feature 15: Comma Density ────────────────────────────────────────────────
#
# Vírgulas por frase. IA cria listas e enumerações com vírgulas de forma
# muito regular. Um valor muito alto indica estrutura de lista artificial.

def compute_comma_density(text: str) -> float:
    """
    Número médio de vírgulas por frase (normalizado para [0, 1] com clip em 5.0).
    IA tende a criar listas com vírgulas repetidamente.
    """
    from nltk.tokenize import sent_tokenize
    sentences = sent_tokenize(text)
    if not sentences:
        return 0.0
    commas_per_sent = [s.count(',') for s in sentences]
    return min(np.mean(commas_per_sent) / 5.0, 1.0)


# ── Feature 16: Unique Trigrams Ratio ────────────────────────────────────────
#
# Trigrams únicos / total de trigrams. Complementa o bigram_repetition_score
# (feature 9) mas em 3-gramas. Textos humanos naturalmente têm mais diversidade
# porque não repetem padrões de 3 palavras.

def compute_unique_trigrams_ratio(text: str) -> float:
    """
    Proporção de trigrams únicos. Quanto mais alto, mais diverso o vocabulário.
    Textos de IA repetitivos → valor mais baixo.
    Retorna valor em [0.0, 1.0].
    """
    words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    if len(words) < 3:
        return 0.0
    trigrams = list(zip(words, words[1:], words[2:]))
    if not trigrams:
        return 0.0
    return len(set(trigrams)) / len(trigrams)


# ── Feature 17: POS Noun Ratio ───────────────────────────────────────────────
#
# Proporção de substantivos (NN, NNS, NNP, NNPS no Penn Treebank tagset).
# Textos de IA tendem a ser mais "substantivados" — usam mais nomes abstratos
# para soar formais: 'application', 'implementation', 'consideration'...

def compute_pos_noun_ratio(text: str) -> float:
    """
    Proporção de substantivos no texto (POS tagging com NLTK).
    IA tende a usar mais substantivos para soar formal.
    Retorna valor em [0.0, 1.0].
    """
    try:
        from nltk import pos_tag, word_tokenize
        # Amostra até 300 palavras para não ser lento demais
        words = word_tokenize(text)[:300]
        if not words:
            return 0.0
        tags  = pos_tag(words)
        nouns = sum(1 for _, t in tags if t.startswith('NN'))
        return nouns / len(words)
    except Exception:
        return 0.0


print("✓ Features 15–17 definidas")

In [ ]:
# ── Feature 18: Coherence Score ──────────────────────────────────────────────
#
# Média da similaridade cosseno TF-IDF entre frases adjacentes.
# IA produz textos mais coerentes (todas as frases falam do mesmo assunto,
# sem desvios), enquanto humanos divagam, mudam de assunto, contradizem-se.
# Um coherence_score muito alto (> 0.6) é sinal de IA.

def compute_coherence_score(text: str) -> float:
    """
    Similaridade TF-IDF cosseno média entre frases adjacentes.
    IA → alta coerência (> 0.5). Humano → baixa/média (< 0.4).
    Retorna valor em [0.0, 1.0].
    """
    try:
        from nltk.tokenize import sent_tokenize
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity

        sents = sent_tokenize(text)
        if len(sents) < 3:
            return 0.0
        # Usa até 20 frases para manter rápido
        sents = sents[:20]
        vec = TfidfVectorizer(max_features=200, stop_words='english')
        tfidf = vec.fit_transform(sents)
        sims  = [
            cosine_similarity(tfidf[i], tfidf[i+1])[0][0]
            for i in range(len(sents) - 1)
        ]
        return float(np.mean(sims))
    except Exception:
        return 0.0


# ── Feature 19: Exclamation Ratio ────────────────────────────────────────────
#
# Proporção de frases que terminam com '!'. Humanos são mais expressivos e
# emocionais, usando exclamações para dar ênfase. IA raramente usa '!'.

def compute_exclamation_ratio(text: str) -> float:
    """
    Proporção de frases exclamativas no texto.
    Humanos → levemente mais alto. IA → quase zero (textos formais/neutros).
    Retorna valor em [0.0, 1.0].
    """
    from nltk.tokenize import sent_tokenize
    sentences = sent_tokenize(text)
    if not sentences:
        return 0.0
    exclamations = sum(1 for s in sentences if s.strip().endswith('!'))
    return exclamations / len(sentences)


print("✓ Features 18–19 definidas")

# ── Nomes completos das features v4 ─────────────────────────────────────────
FEATURE_NAMES_V4 = FEATURE_NAMES + [
    "readability_fog",          # 12
    "stopword_ratio",           # 13
    "sentence_length_variance", # 14
    "comma_density",            # 15
    "unique_trigrams_ratio",    # 16
    "pos_noun_ratio",           # 17
    "coherence_score",          # 18
    "exclamation_ratio",        # 19
]
N_FEATURES_V4 = len(FEATURE_NAMES_V4)
print(f"\nTotal de features v4: {N_FEATURES_V4}")
print("Novas features (12→19):")
for i in range(12, N_FEATURES_V4):
    print(f"  {i:2d}. {FEATURE_NAMES_V4[i]}")

---
## Seção 5 — Extração Paralela com CPU Multi-core

Extraímos as 20 features de todo o dataset usando `joblib.Parallel`.
Com 8 cores de CPU, o processamento de 300k textos leva ~20-40 minutos.

**Estratégia de progresso:** Dividimos em batches de 10k e salvamos o checkpoint
em disco. Se a extração for interrompida, não precisamos recomeçar do zero.

In [ ]:
# ── Função extratora completa (20 features) ───────────────────────────────────

def extract_features_v4(text: str) -> list[float]:
    """
    Extrai as 20 features do modelo v4:
    - Features 0-11: heurísticas do modelo v3 (já validadas)
    - Features 12-19: novas features estilométricas/linguísticas
    """
    # Features v3 (0-11)
    v3 = _extract_safe(text)
    # Novas features (12-19)
    new_feats = [
        compute_readability_fog(text),           # 12
        compute_stopword_ratio(text),            # 13
        compute_sentence_length_variance(text),  # 14
        compute_comma_density(text),             # 15
        compute_unique_trigrams_ratio(text),     # 16
        compute_pos_noun_ratio(text),            # 17
        compute_coherence_score(text),           # 18
        compute_exclamation_ratio(text),         # 19
    ]
    return v3 + new_feats


# ── Teste rápido nas 2 amostras canônicas ────────────────────────────────────

AI_SAMPLE = (
    "Artificial intelligence has fundamentally transformed the landscape of modern technology. "
    "Furthermore, the systematic application of machine learning algorithms enables unprecedented "
    "analytical capabilities. Moreover, neural network architectures facilitate sophisticated "
    "pattern recognition. It is important to note that these advancements have significant "
    "implications for various industries. Consequently, organizations must adapt accordingly."
)
HUMAN_SAMPLE = (
    "I was really struggling with this problem all week. Maybe I'm overthinking it? "
    "My colleague suggested a different approach, and honestly, I think she might be right. "
    "I tried it yesterday and it sort of worked? Not perfectly, but better. "
    "Perhaps if I tweak a few things it'll click. Does anyone else deal with this kind of thing?"
)

f_ai    = extract_features_v4(AI_SAMPLE)
f_human = extract_features_v4(HUMAN_SAMPLE)

print(f"{'Feature':<30} {'IA':>8} {'Humano':>8} {'Delta':>8}")
print('-' * 58)
for i, name in enumerate(FEATURE_NAMES_V4):
    marker = ' ◄' if i >= 12 else ''
    print(f"{name:<30} {f_ai[i]:>8.4f} {f_human[i]:>8.4f} {f_ai[i]-f_human[i]:>+8.4f}{marker}")

In [ ]:
# ── Extração paralela do dataset completo ────────────────────────────────────
#
# Checkpoint: salva arrays numpy a cada batch para poder retomar.
# n_jobs=-1 usa todos os cores disponíveis.

CHECKPOINT_PATH = MODELS_DIR / 'v4_features_checkpoint.npz'
BATCH_SIZE = 10_000

if CHECKPOINT_PATH.exists():
    print("♻️  Checkpoint encontrado — carregando features salvas...")
    ckpt  = np.load(CHECKPOINT_PATH)
    X_v4  = ckpt['X'].astype(np.float32)
    y_v4  = ckpt['y']
    print(f"   Carregados: {X_v4.shape[0]:,} amostras × {X_v4.shape[1]} features")
else:
    texts  = df_all['text'].tolist()
    labels = df_all['label'].values
    n      = len(texts)
    n_batches = (n + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"Extraindo {N_FEATURES_V4} features de {n:,} textos em {n_batches} batches...")
    print(f"Usando todos os cores de CPU disponíveis (n_jobs=-1)")

    all_feats = []
    t0 = time.time()

    for i in range(n_batches):
        start = i * BATCH_SIZE
        end   = min(start + BATCH_SIZE, n)
        batch = texts[start:end]

        batch_feats = Parallel(n_jobs=-1, prefer='processes')(
            delayed(extract_features_v4)(t) for t in batch
        )
        all_feats.extend(batch_feats)

        elapsed = time.time() - t0
        pct     = end / n * 100
        eta     = elapsed / pct * (100 - pct) if pct > 0 else 0
        print(f"  Batch {i+1}/{n_batches}: {end:,}/{n:,}  ({pct:.1f}%)  ETA: {eta:.0f}s")

    X_v4 = np.array(all_feats, dtype=np.float32)
    y_v4 = labels

    # Salva checkpoint
    np.savez_compressed(CHECKPOINT_PATH, X=X_v4, y=y_v4)
    total_time = time.time() - t0
    print(f"\n✓ Extração concluída em {total_time/60:.1f} minutos")
    print(f"✓ Checkpoint salvo: {CHECKPOINT_PATH}")

print(f"\nShape final: X={X_v4.shape}, y={y_v4.shape}")
print(f"NaN check: {np.isnan(X_v4).sum()} NaNs")
X_v4 = np.nan_to_num(X_v4, nan=0.0, posinf=1.0, neginf=0.0)

---
## Seção 6 — Seleção e Análise de Features

Antes de treinar, entendemos quais features são mais informativas:
- **Distribuição por classe**: como cada feature separa IA de humano
- **Correlação**: features altamente correlacionadas não adicionam informação
- **Importância preliminar**: Random Forest rápido para ranking inicial

In [ ]:
# ── Distribuição de cada feature por classe ──────────────────────────────────

df_feats = pd.DataFrame(X_v4, columns=FEATURE_NAMES_V4)
df_feats['label'] = y_v4

# Plota violin plots para as features mais importantes
TOP_FEATURES = ['burstiness', 'coherence_score', 'transition_word_density',
                'first_person_ratio', 'stopword_ratio', 'pos_noun_ratio',
                'readability_fog', 'unique_trigrams_ratio']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(TOP_FEATURES):
    ax = axes[i]
    data_h = df_feats[df_feats['label']==0][feat].values
    data_a = df_feats[df_feats['label']==1][feat].values
    ax.violinplot([data_h, data_a], positions=[0, 1],
                  showmedians=True, showmeans=False)
    ax.set_title(feat, fontsize=9, fontweight='bold')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Human', 'AI'])
    ax.set_ylabel('Valor')

plt.suptitle('Distribuição das Features por Classe (Human vs IA)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Matriz de correlação ─────────────────────────────────────────────────────
#
# Features altamente correlacionadas (|r| > 0.85) são redundantes.
# Identificamos e podemos remover as menos importantes do par.

# Amostramos para calcular correlação mais rápido
sample_idx = np.random.choice(len(X_v4), min(30000, len(X_v4)), replace=False)
corr_matrix = pd.DataFrame(X_v4[sample_idx], columns=FEATURE_NAMES_V4).corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            annot_kws={'size': 7})
ax.set_title('Matriz de Correlação — 20 Features v4', fontsize=12)
plt.tight_layout()
plt.show()

# Identifica pares altamente correlacionados
high_corr = []
for i in range(N_FEATURES_V4):
    for j in range(i+1, N_FEATURES_V4):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.75:
            high_corr.append((FEATURE_NAMES_V4[i], FEATURE_NAMES_V4[j], r))

if high_corr:
    print("Pares com correlação alta (|r| > 0.75):")
    for f1, f2, r in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
        print(f"  {f1:30s} × {f2:30s}  r={r:+.3f}")
else:
    print("✓ Nenhum par com correlação > 0.75 — todas as features são informativas.")

In [ ]:
# ── Importância prévia: Random Forest rápido ─────────────────────────────────
#
# Treinamos um RF rápido (50 árvores) só para ter o ranking de importância
# antes do treinamento completo. Isso guia a interpretação.

X_tr_pre, _, y_tr_pre, _ = train_test_split(
    X_v4, y_v4, test_size=0.2, random_state=SEED, stratify=y_v4
)
rf_quick = RandomForestClassifier(n_estimators=50, max_depth=8,
                                   random_state=SEED, n_jobs=-1)
rf_quick.fit(X_tr_pre, y_tr_pre)

importances = rf_quick.feature_importances_
idx_sorted  = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#ef4444' if i < 12 else '#6366f1' for i in idx_sorted]
ax.barh([FEATURE_NAMES_V4[i] for i in idx_sorted[::-1]],
         importances[idx_sorted[::-1]], color=colors[::-1], alpha=0.85)
ax.set_xlabel('Importância (Gini)', fontsize=11)
ax.set_title('Importância das Features — RF rápido (50 árvores)', fontsize=12)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#ef4444', label='Features v3 (0–11)'),
    Patch(facecolor='#6366f1', label='Features v4 novas (12–19)'),
], loc='lower right')
plt.tight_layout(); plt.show()

print("\nRanking de importância:")
for rank, i in enumerate(idx_sorted):
    tag = '🆕' if i >= 12 else '  '
    print(f"  {rank+1:2d}. {tag} {FEATURE_NAMES_V4[i]:<30s}: {importances[i]:.4f}")

---
## Seção 7 — Treinamento: XGBoost + LightGBM com GPU

Testamos dois modelos de boosting que tipicamente superam Random Forest:

- **XGBoost** (`device='cuda'`): Gradient boosting com regularização L1/L2. Excelente em tabular data. GPU acelera ~10x.
- **LightGBM** (`device='gpu'`): Mais rápido que XGBoost em datasets grandes, usa histogram-based splitting. GPU acelera ~5-8x.

Ambos são avaliados com validação cruzada estratificada de 5 folds.

In [ ]:
# ── Split treino/teste (80/20 estratificado) ─────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X_v4, y_v4, test_size=0.20, random_state=SEED, stratify=y_v4
)

print(f"Treino: {len(X_train):,}  |  Teste: {len(X_test):,}")
print(f"Balanço treino: {y_train.mean():.3f} (IA)  |  Teste: {y_test.mean():.3f} (IA)")

In [ ]:
# ── XGBoost com GPU ──────────────────────────────────────────────────────────
#
# Parâmetros iniciais razoáveis. A Seção 8 (Optuna) vai otimizá-los.
# scale_pos_weight: compensa desbalanço de classes se existir.

scale_pw = (y_train == 0).sum() / (y_train == 1).sum()  # razão human/ai

xgb_model = xgb.XGBClassifier(
    n_estimators     = 500,
    learning_rate    = 0.05,
    max_depth        = 7,
    subsample        = 0.85,
    colsample_bytree = 0.85,
    reg_alpha        = 0.1,    # L1 regularização
    reg_lambda       = 1.0,    # L2 regularização
    scale_pos_weight = scale_pw,
    tree_method      = 'hist',
    device           = XGB_DEVICE,
    eval_metric      = 'logloss',
    early_stopping_rounds = 30,
    random_state     = SEED,
    verbosity        = 0,
)

print(f"Treinando XGBoost (device={XGB_DEVICE})...")
t0 = time.time()

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

elapsed = time.time() - t0
print(f"✓ XGBoost treinado em {elapsed:.1f}s  |  {xgb_model.best_iteration} árvores")

y_pred_xgb  = xgb_model.predict(X_test)
y_prob_xgb  = xgb_model.predict_proba(X_test)[:, 1]
acc_xgb     = accuracy_score(y_test, y_pred_xgb)
f1_xgb      = f1_score(y_test, y_pred_xgb, average='weighted')
auc_xgb     = roc_auc_score(y_test, y_prob_xgb)

print(f"\n  XGBoost → Accuracy: {acc_xgb:.4f} | F1: {f1_xgb:.4f} | AUC: {auc_xgb:.4f}")

In [ ]:
# ── LightGBM com GPU ─────────────────────────────────────────────────────────

lgb_model = lgb.LGBMClassifier(
    n_estimators     = 500,
    learning_rate    = 0.05,
    num_leaves       = 63,
    max_depth        = -1,
    subsample        = 0.85,
    colsample_bytree = 0.85,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    device           = LGB_DEVICE,
    class_weight     = 'balanced',
    random_state     = SEED,
    verbose          = -1,
)

print(f"Treinando LightGBM (device={LGB_DEVICE})...")
t0 = time.time()

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)],
)

elapsed = time.time() - t0
print(f"✓ LightGBM treinado em {elapsed:.1f}s")

y_pred_lgb = lgb_model.predict(X_test)
y_prob_lgb = lgb_model.predict_proba(X_test)[:, 1]
acc_lgb    = accuracy_score(y_test, y_pred_lgb)
f1_lgb     = f1_score(y_test, y_pred_lgb, average='weighted')
auc_lgb    = roc_auc_score(y_test, y_prob_lgb)

print(f"\n  LightGBM → Accuracy: {acc_lgb:.4f} | F1: {f1_lgb:.4f} | AUC: {auc_lgb:.4f}")

print("\n──────────────────────────────────────────")
if acc_v3 > 0:
    print(f"  Baseline v3 RF: {acc_v3:.4f}")
print(f"  XGBoost v4:     {acc_xgb:.4f}  ({acc_xgb-acc_v3:+.4f})")
print(f"  LightGBM v4:    {acc_lgb:.4f}  ({acc_lgb-acc_v3:+.4f})")
print("──────────────────────────────────────────")

---
## Seção 8 — Optuna: Busca Automática de Hiperparâmetros

**Optuna** é um framework de otimização bayesiana que encontra os melhores
hiperparâmetros de forma muito mais eficiente que grid search.

Rodamos **100 trials** no modelo vencedor (XGBoost ou LightGBM).
Cada trial treina um modelo com parâmetros diferentes e avalia no conjunto de validação.

In [ ]:
# ── Split adicional: validação para Optuna ───────────────────────────────────
#
# Separamos 10% do treino como validação para o Optuna,
# mantendo o teste (20%) intocado para a avaliação final.

X_tr_opt, X_val_opt, y_tr_opt, y_val_opt = train_test_split(
    X_train, y_train, test_size=0.125, random_state=SEED, stratify=y_train
)
print(f"Optuna — Treino: {len(X_tr_opt):,}  |  Val: {len(X_val_opt):,}  |  Teste: {len(X_test):,}")

In [ ]:
# ── Objetivo Optuna para XGBoost ─────────────────────────────────────────────

def xgb_objective(trial: optuna.Trial) -> float:
    """Função objetivo: maximiza AUC-ROC no conjunto de validação."""
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth'        : trial.suggest_int('max_depth', 4, 10),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'min_child_weight' : trial.suggest_int('min_child_weight', 1, 10),
        'gamma'            : trial.suggest_float('gamma', 0, 5),
    }
    model = xgb.XGBClassifier(
        **params,
        tree_method='hist',
        device=XGB_DEVICE,
        random_state=SEED,
        verbosity=0,
        early_stopping_rounds=20,
        eval_metric='logloss',
    )
    model.fit(X_tr_opt, y_tr_opt,
              eval_set=[(X_val_opt, y_val_opt)],
              verbose=False)
    probs = model.predict_proba(X_val_opt)[:, 1]
    return roc_auc_score(y_val_opt, probs)


print("Iniciando busca de hiperparâmetros XGBoost (100 trials)...")
print("Isso pode levar 10-30 minutos dependendo do hardware.")

study_xgb = optuna.create_study(direction='maximize',
                                  study_name='xgb_v4_detector')
study_xgb.optimize(xgb_objective, n_trials=100,
                    show_progress_bar=True, n_jobs=1)

print(f"\n✓ Melhor AUC-ROC (validação): {study_xgb.best_value:.5f}")
print("Melhores hiperparâmetros:")
for k, v in study_xgb.best_params.items():
    print(f"  {k:<25s}: {v}")

In [ ]:
# ── Treina XGBoost final com os melhores hiperparâmetros ─────────────────────

best_params = study_xgb.best_params.copy()
# Remove early_stopping_rounds dos params (vai via fit)
n_est_final = best_params.pop('n_estimators')

xgb_best = xgb.XGBClassifier(
    **best_params,
    n_estimators=n_est_final,
    tree_method='hist',
    device=XGB_DEVICE,
    eval_metric='logloss',
    random_state=SEED,
    verbosity=0,
)

print("Treinando XGBoost otimizado no conjunto COMPLETO de treino...")
t0 = time.time()
xgb_best.fit(X_train, y_train)
print(f"✓ Treinado em {time.time()-t0:.1f}s")

y_pred_best = xgb_best.predict(X_test)
y_prob_best = xgb_best.predict_proba(X_test)[:, 1]

acc_best = accuracy_score(y_test, y_pred_best)
f1_best  = f1_score(y_test, y_pred_best, average='weighted')
auc_best = roc_auc_score(y_test, y_prob_best)

print(f"\n  XGBoost otimizado → Accuracy: {acc_best:.4f} | F1: {f1_best:.4f} | AUC: {auc_best:.4f}")

---
## Seção 9 — Ensemble Calibrado

Combinamos XGBoost + LightGBM + Random Forest em um **VotingClassifier** com
votação suave (média de probabilidades), depois aplicamos **calibração de Platt**
para garantir que os scores de probabilidade sejam bem calibrados.

**Calibração** é fundamental: queremos que "score 0.80" realmente signifique
80% de chance de ser IA, não apenas uma pontuação relativa.

In [ ]:
# ── Random Forest v4 para o ensemble ─────────────────────────────────────────

rf_v4 = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_leaf=3,
    class_weight='balanced', random_state=SEED, n_jobs=-1
)
print("Treinando RF v4...")
rf_v4.fit(X_train, y_train)

y_pred_rf4 = rf_v4.predict(X_test)
y_prob_rf4 = rf_v4.predict_proba(X_test)[:, 1]
acc_rf4    = accuracy_score(y_test, y_pred_rf4)
print(f"  RF v4 → Accuracy: {acc_rf4:.4f}")

In [ ]:
# ── VotingClassifier (soft voting) ───────────────────────────────────────────
#
# Soft voting = média ponderada das probabilidades de cada modelo.
# Damos mais peso ao XGBoost otimizado (2) vs RF e LGB (1 cada).

ensemble = VotingClassifier(
    estimators=[
        ('xgb', xgb_best),
        ('lgb', lgb_model),
        ('rf',  rf_v4),
    ],
    voting='soft',
    weights=[2, 1, 1],  # peso maior para XGBoost otimizado
    n_jobs=-1,
)

print("Treinando ensemble (soft voting)...")
t0 = time.time()
ensemble.fit(X_train, y_train)
print(f"✓ Ensemble treinado em {time.time()-t0:.1f}s")

y_pred_ens = ensemble.predict(X_test)
y_prob_ens = ensemble.predict_proba(X_test)[:, 1]
acc_ens    = accuracy_score(y_test, y_pred_ens)
f1_ens     = f1_score(y_test, y_pred_ens, average='weighted')
auc_ens    = roc_auc_score(y_test, y_prob_ens)

print(f"\n  Ensemble → Accuracy: {acc_ens:.4f} | F1: {f1_ens:.4f} | AUC: {auc_ens:.4f}")

In [ ]:
# ── Calibração de Platt ───────────────────────────────────────────────────────
#
# A calibração de Platt ajusta os scores para que sejam probabilidades reais.
# Usamos CalibratedClassifierCV com cv='prefit' (já temos o modelo treinado).

# Seleciona o melhor modelo individual para calibrar
best_individual = xgb_best if acc_best >= acc_lgb else lgb_model
best_individual_name = 'XGBoost' if acc_best >= acc_lgb else 'LightGBM'

calibrated = CalibratedClassifierCV(
    best_individual,
    method='sigmoid',   # Platt scaling
    cv='prefit',
)

# Calibramos no conjunto de validação (X_val_opt)
print(f"Calibrando {best_individual_name}...")
calibrated.fit(X_val_opt, y_val_opt)

y_prob_cal = calibrated.predict_proba(X_test)[:, 1]
y_pred_cal = (y_prob_cal >= 0.5).astype(int)
acc_cal    = accuracy_score(y_test, y_pred_cal)
brier_cal  = brier_score_loss(y_test, y_prob_cal)

print(f"  Calibrado → Accuracy: {acc_cal:.4f} | Brier Score: {brier_cal:.4f} (menor = melhor calibração)")

---
## Seção 10 — Avaliação Completa

Comparação final de todos os modelos com métricas completas:
accuracy, F1, AUC-ROC, curva de calibração e matriz de confusão.

In [ ]:
# ── Tabela comparativa de todos os modelos ────────────────────────────────────

models_eval = [
    ('v3 RF (baseline)',   y_pred_v3,  y_prob_v3)   if acc_v3 > 0 else None,
    ('XGBoost v4',         y_pred_xgb, y_prob_xgb),
    ('LightGBM v4',        y_pred_lgb, y_prob_lgb),
    ('XGBoost otimizado',  y_pred_best, y_prob_best),
    ('Ensemble (soft)',    y_pred_ens, y_prob_ens),
    (f'{best_individual_name} calibrado', y_pred_cal, y_prob_cal),
]
models_eval = [m for m in models_eval if m is not None]

print(f"{'Modelo':<30} {'Accuracy':>10} {'F1 wtd':>10} {'ROC-AUC':>10} {'Brier':>8}")
print('─' * 72)
best_row = None; best_acc = 0
for name, preds, probs in models_eval:
    acc  = accuracy_score(y_test, preds)
    f1   = f1_score(y_test, preds, average='weighted')
    auc  = roc_auc_score(y_test, probs)
    brier= brier_score_loss(y_test, probs)
    flag = ' ◄' if acc > best_acc else ''
    print(f"{name:<30} {acc:>10.4f} {f1:>10.4f} {auc:>10.4f} {brier:>8.4f}{flag}")
    if acc > best_acc:
        best_acc = acc
        best_row = (name, preds, probs)

print('─' * 72)
print(f"\n🏆 Melhor modelo: {best_row[0]}  ({best_acc*100:.2f}% accuracy)")

In [ ]:
# ── Curvas ROC e Calibração ───────────────────────────────────────────────────

from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
for name, preds, probs in models_eval:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_val = roc_auc_score(y_test, probs)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})")
axes[0].plot([0,1],[0,1],'k--', alpha=0.3, label='Random')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('Curva ROC — Comparação de Modelos')
axes[0].legend(fontsize=8, loc='lower right')

# Calibração
for name, preds, probs in models_eval:
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=10)
    axes[1].plot(mean_pred, frac_pos, marker='o', markersize=3, label=name)
axes[1].plot([0,1],[0,1],'k--', alpha=0.5, label='Calibração perfeita')
axes[1].set_xlabel('Score médio predito')
axes[1].set_ylabel('Fração positiva real')
axes[1].set_title('Curva de Calibração')
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
# ── Matriz de confusão do melhor modelo ─────────────────────────────────────

best_name, best_preds, best_probs = best_row

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_test, best_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['Human', 'AI'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Matriz de Confusão\n{best_name}')

# Distribuição dos scores de probabilidade
axes[1].hist(best_probs[y_test==0], bins=50, alpha=0.7, label='Human', color='#22c55e', density=True)
axes[1].hist(best_probs[y_test==1], bins=50, alpha=0.7, label='AI',    color='#ef4444', density=True)
axes[1].axvline(0.5, color='gray', linestyle='--', label='Threshold=0.5')
axes[1].set_xlabel('P(AI)')
axes[1].set_ylabel('Densidade')
axes[1].set_title('Distribuição de Scores')
axes[1].legend()

plt.tight_layout(); plt.show()
print(classification_report(y_test, best_preds, target_names=['Human','AI']))

---
## Seção 11 — Análise de Erros

Os erros mais informativos são:
- **Falsos positivos**: textos humanos classificados como IA (risco de injustiça)
- **Falsos negativos**: textos de IA não detectados (risco de evasão)

Examinamos os casos de alta confiança errada para entender os pontos cegos do modelo.

In [ ]:
# ── Análise de erros de alta confiança ───────────────────────────────────────

# Subset de teste com o texto original
test_idx  = df_all.index[len(X_train):len(X_train)+len(X_test)]
df_test   = df_all.iloc[:len(X_test)].copy() if len(test_idx) != len(X_test) else df_all.loc[test_idx].copy()

# Reconstrói o df de teste pelo split
_, df_test_texts = train_test_split(df_all, test_size=0.20, random_state=SEED, stratify=y_v4)
df_test_texts = df_test_texts.reset_index(drop=True)

df_test_texts['pred_prob'] = best_probs
df_test_texts['pred']      = best_preds
df_test_texts['correct']   = (df_test_texts['pred'] == df_test_texts['label'])
df_test_texts['confidence']= df_test_texts['pred_prob'].apply(
    lambda p: abs(p - 0.5) * 2  # 0 = total incerteza, 1 = certeza máxima
)

# Falsos positivos com alta confiança (humano classificado como IA com certeza)
fp_high = df_test_texts[
    (df_test_texts['label']==0) &
    (df_test_texts['pred']==1) &
    (df_test_texts['confidence'] > 0.7)
].nlargest(5, 'confidence')

print("━━━━ FALSOS POSITIVOS (humanos marcados como IA, alta confiança) ━━━━")
for _, row in fp_high.iterrows():
    print(f"  Score: {row['pred_prob']:.3f} | Fonte: {row.get('source','?')}")
    print(f"  Texto: {row['text'][:200]}...")
    print()

# Falsos negativos com alta confiança (IA não detectada com certeza)
fn_high = df_test_texts[
    (df_test_texts['label']==1) &
    (df_test_texts['pred']==0) &
    (df_test_texts['confidence'] > 0.7)
].nlargest(5, 'confidence')

print("━━━━ FALSOS NEGATIVOS (IA não detectada, alta confiança) ━━━━")
for _, row in fn_high.iterrows():
    print(f"  Score: {row['pred_prob']:.3f} | Fonte: {row.get('source','?')}")
    print(f"  Texto: {row['text'][:200]}...")
    print()

---
## Seção 12 — Salvar Modelo v4 e Atualizar o Serviço

Salvamos o modelo vencedor e atualizamos `detection_service.py` para usar v4.
O serviço detecta automaticamente o modelo disponível e as features correspondentes.

In [ ]:
# ── Salva modelo v4 ───────────────────────────────────────────────────────────

V4_MODEL_PATH   = MODELS_DIR / 'detector_v4_best.joblib'
V4_METRICS_PATH = MODELS_DIR / 'v4_metrics.json'
V4_FEATURES_PATH= MODELS_DIR / 'v4_feature_names.json'

# Seleciona o modelo para deploy (melhor accuracy no teste)
# Preferimos o calibrado se tiver accuracy comparável
DEPLOY_MODEL = calibrated  # ou ensemble, dependendo dos resultados acima
DEPLOY_ACC   = acc_cal

joblib.dump(DEPLOY_MODEL, V4_MODEL_PATH, compress=3)
print(f"✓ Modelo salvo: {V4_MODEL_PATH}  ({V4_MODEL_PATH.stat().st_size/1024/1024:.1f} MB)")

# Salva lista de feature names para o serviço
with open(V4_FEATURES_PATH, 'w') as f:
    json.dump(FEATURE_NAMES_V4, f, indent=2)
print(f"✓ Feature names salvas: {V4_FEATURES_PATH}")

# Salva métricas
metrics = {
    'model_version'     : 'v4',
    'n_features'        : N_FEATURES_V4,
    'feature_names'     : FEATURE_NAMES_V4,
    'n_train_samples'   : len(X_train),
    'n_test_samples'    : len(X_test),
    'datasets_used'     : ['raid','ai-detection-pile','hc3','mage'],
    'deploy_model'      : 'CalibratedXGBoost' if 'calibr' in best_name.lower() else best_name,
    'test_accuracy'     : float(DEPLOY_ACC),
    'test_f1_weighted'  : float(f1_score(y_test, y_pred_cal, average='weighted')),
    'test_roc_auc'      : float(roc_auc_score(y_test, y_prob_cal)),
    'test_brier'        : float(brier_score_loss(y_test, y_prob_cal)),
    'baseline_v3_acc'   : float(acc_v3) if acc_v3 > 0 else None,
    'improvement_vs_v3' : float(DEPLOY_ACC - acc_v3) if acc_v3 > 0 else None,
    'optuna_best_auc'   : study_xgb.best_value,
    'optuna_best_params': study_xgb.best_params,
}
with open(V4_METRICS_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"✓ Métricas salvas: {V4_METRICS_PATH}")

print(f"\n{'='*55}")
print(f"  RESULTADO FINAL")
print(f"{'='*55}")
if acc_v3 > 0:
    print(f"  Baseline v3:    {acc_v3*100:.2f}%")
print(f"  Modelo v4:      {DEPLOY_ACC*100:.2f}%")
if acc_v3 > 0:
    gain = (DEPLOY_ACC - acc_v3) * 100
    print(f"  Melhora:        {gain:+.2f} pontos percentuais")
print(f"{'='*55}")

In [ ]:
# ── Gera código de atualização do detection_service.py ───────────────────────
#
# Imprime o bloco de código que deve ser inserido no topo de detection_service.py
# para adicionar as 8 novas features e carregar o modelo v4.

UPDATE_SNIPPET = '''
# ── Coloque este bloco no início de detection_service.py ─────────────────────
# Adicione estas 8 funções após compute_hapax_legomena_ratio()
# e atualize extract_features() para chamar todas as 20.
#
# 1. Atualize N_FEATURES_V3 = 12  →  N_FEATURES_V4 = 20
# 2. Atualize _RF_V3_PATH para apontar para detector_v4_best.joblib
# 3. Adicione as 8 funções compute_* deste notebook
# 4. No extract_features(), adicione as 8 chamadas novas
# 5. Os testes em test_new_features.py precisam ser atualizados
'''
print(UPDATE_SNIPPET)

# Resumo final das features novas com seus valores no texto canônico
print("Validação das novas features no texto de IA vs Humano:")
print(f"{'Feature':<30} {'IA':>8} {'Humano':>8} {'Direção esperada'}")
print('-'*75)
expected = ['IA>Human','Human>IA','Human>IA','IA>Human','Human>IA','IA>Human','IA>Human','Human>IA']
for i in range(12, N_FEATURES_V4):
    name = FEATURE_NAMES_V4[i]
    vai  = f_ai[i]
    vhu  = f_human[i]
    exp  = expected[i-12]
    ok   = (vai > vhu) == (exp == 'IA>Human')
    mark = '✓' if ok else '✗'
    print(f"{name:<30} {vai:>8.4f} {vhu:>8.4f}  {mark} {exp}")

In [ ]:
# ── Diagrama final de evolução do modelo ─────────────────────────────────────

versions = ['v1\n(heurístico)', 'v2\n(HC3 5-feat)', 'v3\n(RF 12-feat)', 'v4\n(XGB 20-feat)']
accs     = [0.72, 0.88, 0.934, DEPLOY_ACC]
colors   = ['#94a3b8', '#60a5fa', '#6366f1', '#22c55e']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(versions, [a*100 for a in accs], color=colors, alpha=0.9,
              width=0.5, edgecolor='white', linewidth=1.5)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{acc*100:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_ylim(60, 100)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Evolução do Modelo AI Detector', fontsize=14, fontweight='bold')
ax.axhline(93.4, color='gray', linestyle='--', alpha=0.5, label='Baseline v3 (93.4%)')
ax.axhline(96.0, color='#22c55e', linestyle=':', alpha=0.5, label='Meta v4 (96%)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("\n🎉 Evolução concluída! Modelo v4 pronto para integração.")